# Build & Diagnose a Predictive Organism

End-to-end: record a nursery world, gate its quality, train the Predictive
Cortex, prove it beats copy-last (Milestone 2), prove it uses actions,
probe the representation, dream from the hippocampus (Milestone 4), measure
forgetting under generative replay (Milestone 5), and export everything for
the clinic.

See `docs/v2/REVIEW-2026-07-organism-audit.md` for what each step proves.
Step 9 (issue #169) checks the reward/terminal/risk/uncertainty heads --
trained since `train_action_world_model` now optimizes them alongside
pixel+latent -- against a constant-predictor baseline and the
uncertainty/realized-error correlation.

> Requires the `neural` and `crafter` extras: `pip install -e .[neural,crafter]`.

## 1. Setup

In [ ]:
import os, pathlib
import matplotlib.pyplot as plt

ORGANISM = "Pixel"          # names the model id and every file it generates
WORLD = "crafter"           # the deterministic pixel-native nursery
RECORD_DIR = pathlib.Path("runs") / ORGANISM
RECORD_DIR.mkdir(parents=True, exist_ok=True)
HORIZONS_TICKS = (1, 4, 8)
SCENARIOS = ["walk_forward", "turn", "approach_entity", "object_permanence"]

## 2. Record nursery data (train + holdout)

`run_nursery_scenario` records train and holdout episodes for one scenario in
Crafter with pixel frames. Equivalent CLI: `ccr nursery run <scenario> --world
crafter --record-frames --name Pixel`.

In [ ]:
from cognitive_runtime.training.nursery import (
    NurseryConfig, run_nursery_scenario, _scenarios_for_world,
)

cfg = NurseryConfig(world=WORLD, name=ORGANISM, export_predictions=True)
train_dirs, holdout_dirs = [], []
for name in SCENARIOS:
    _model, report = run_nursery_scenario(str(RECORD_DIR), name, cfg)
    train_dirs.extend(report.train_sessions)
    holdout_dirs.extend(report.holdout_sessions)
print("train:", train_dirs)
print("holdout:", holdout_dirs)

## 3. Data-quality gates — halt on red

No phase trains on a red session (pixel provenance, motion floor, completed
episodes, frozen recording).

In [ ]:
from cognitive_runtime.record.quality import evaluate_record_quality

for d in train_dirs + holdout_dirs:
    verdict = evaluate_record_quality(d)
    print(f"{verdict.verdict:>5}  {d}  {verdict.issues}")
    assert verdict.verdict != "red", f"refusing to train on a red session: {d}"

## 4. Build the joint action-conditioned dataset

In [ ]:
from cognitive_runtime.training.action_world_model import build_action_sequence_dataset
from cognitive_runtime.programs.crafter.actions import ACTION_SPACE  # pin full vocab

action_keys = [a.name for a in ACTION_SPACE]
train_ds = build_action_sequence_dataset(train_dirs, action_keys=action_keys)
holdout_ds = build_action_sequence_dataset(holdout_dirs, action_keys=action_keys)
print("train transitions:", len(train_ds), "| pixel shape:", train_ds.pixel_shape)

## 5. Train the Predictive Cortex + loss curves

In [ ]:
from cognitive_runtime.training.action_world_model import (
    ActionWorldModelConfig, train_action_world_model,
)

from dataclasses import replace

model_cfg = ActionWorldModelConfig(horizons_ticks=HORIZONS_TICKS, epochs=30, backbone="gru", ema_target_decay=0.99)
baseline_model, baseline_stats = train_action_world_model(train_ds, replace(model_cfg, ema_target_decay=None))
model, stats = train_action_world_model(train_ds, model_cfg)

for key in ("total_loss", "pixel_loss", "latent_loss"):
    plt.plot(stats["loss_curves"][key], label=key)
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("cortex training"); plt.show()

## 6. Milestone 2a — beat copy-last at every horizon + frozen-rollout check

In [ ]:
from cognitive_runtime.training.action_world_model import (
    evaluate_action_world_model, horizons_ticks_to_frames,
)

hf = horizons_ticks_to_frames(HORIZONS_TICKS, holdout_ds.ticks_per_frame)
ev = evaluate_action_world_model(model, holdout_ds, hf)
baseline_ev = evaluate_action_world_model(baseline_model, holdout_ds, hf)
ema_mse = sum(r["model_mse"] for r in ev["horizons"].values()) / len(hf)
baseline_mse = sum(r["model_mse"] for r in baseline_ev["horizons"].values()) / len(hf)
print(f"EMA held-out MSE={ema_mse:.6f}; shared-encoder baseline={baseline_mse:.6f}")
assert ema_mse <= baseline_mse, "EMA target regressed held-out prediction"
for h, row in ev["horizons"].items():
    print(f"t+{h}: model/copy-last={row['model_over_copy_last_mse']:.3f} "
          f"beats={row['beats_copy_last']} PSNR={row['psnr_model']:.1f}dB")
assert ev["rollout_health"]["frozen_rollout"] is False, "cortex collapsed to a fixed point"
assert all(r["beats_copy_last"] for r in ev["horizons"].values()), "Milestone 2a failed"

## 7. Milestone 2b — withholding actions must hurt

In [ ]:
from cognitive_runtime.training.nursery import run_action_ablation_eval

abl = run_action_ablation_eval(train_dirs, holdout_dirs, action_keys=action_keys,
                               horizons_ticks=HORIZONS_TICKS)
print(abl)  # expect: with-actions MSE < without-actions MSE (the model uses actions)

## 8. Representation probe — does the hidden state decode heading?

A hidden state that linearly decodes yaw where the raw latent cannot is
evidence the recurrence carries motion state — and a guard against latent
collapse (see audit bug (c)).

In [ ]:
from cognitive_runtime.training.action_world_model import representation_collapse_diagnostics

representation = representation_collapse_diagnostics(model, holdout_ds, config=model_cfg)
print(representation)
if representation["gate_evaluable"]:
    assert representation["passed"], "cortex representation collapsed; do not promote"
else:
    print("representation gate N/A: too few yaw-labelled holdout frames")

## 9. Heads diagnostic

The cortex's reward/terminal/risk/uncertainty heads are trained (issue
#169: `train_action_world_model` adds reward MSE, terminal BCE, risk MSE,
and an uncertainty-vs-realized-latent-error term to pixel+latent). This
cell checks the acceptance criteria on held-out data: each head beats a
constant-predictor baseline, and predicted `uncertainty` correlates
positively with realized latent error (CI clear of 0).

In [ ]:
import torch
from cognitive_runtime.training.action_world_model import _episode_tensors, evaluate_cortex_heads

episodes = _episode_tensors(holdout_ds, model.reconstruction_shape)
heads_report = evaluate_cortex_heads(model, holdout_ds, hf)
for h, row in heads_report.items():
    corr, (ci_lo, ci_hi) = row["uncertainty_error_correlation"], row["uncertainty_error_correlation_ci"]
    print(f"t+{h}: reward beats-const={row['reward_beats_constant']} "
          f"terminal beats-const={row['terminal_beats_constant']} "
          f"risk beats-const={row['risk_beats_constant']} "
          f"uncertainty-error corr={corr:+.3f} (CI {ci_lo:+.3f}..{ci_hi:+.3f})")
    # `beats_constant` is None for a degenerate (zero-variance) target
    # column -- e.g. a holdout split with no death events, where the
    # constant predictor scores an unbeatable exact 0.0 -- so only a
    # literal `False` (the head actually loses to the baseline) fails.
    for head in ("reward", "terminal", "risk"):
        assert row[f"{head}_beats_constant"] is not False, f"t+{h}: {head} head loses to the constant baseline"
    assert ci_lo > 0.0, f"t+{h}: uncertainty/realized-error correlation CI is not clear of 0"

## 10. Milestone 4 — dream from a hippocampal seed, export the strip

In [ ]:
from brain.hippocampus import Hippocampus, SeedTags
from sleep.dream import dream, export_dream_file

ep, pix, tgt, act = episodes[0]
with torch.no_grad():
    z0 = model.encoder(pix[:1]).squeeze(0)
hippo = Hippocampus(capacity=256)
seed = hippo.encode(z=z0.tolist(),
                    actions=[model.action_keys[int(a)] for a in act[: max(hf)]],
                    tags=SeedTags(dopamine=0.0, threat=0.0, novelty=1.0),
                    source=holdout_dirs[0])
actual_frames = [pix[i] for i in range(max(hf) + 1)]
out_path = export_dream_file(seed, model, actual_frames, horizons=list(hf),
                             session_dir=holdout_dirs[0], name=ORGANISM)
print("dream strip:", out_path)

## 11. Milestone 5 — forgetting under generative replay vs flat training

Train scenario A, hold-out loss; train B two ways (flat vs staged+dream replay
via `GenerativeReplayMixer`); does A's accuracy survive? See
`tests/test_forgetting_metric.py` for the reference wiring of both conditions.

In [ ]:
from sleep.forgetting import compute_forgetting_metric
# old_before / old_after are held-out loss samples on scenario A, measured
# before and after training scenario B (populate from evaluate_action_world_model
# per-episode samples under each condition).
old_before = []  # e.g. ev_A['per_episode_model_mse'][hf[0]]
old_after = []   # same, after training B
if old_before and old_after:
    report = compute_forgetting_metric(old_before, old_after,
                                       old_scenario="walk_forward", new_scenario="turn",
                                       tolerance=0.02)
    print("retained:", report.retained, "| forgetting:", report.forgetting_amount)

## 12. Export for the clinic, then inspect

`export_predictions=True` already wrote `Pixel-predictions_*.json` next to each
recorded episode. Serve and open the clinic:

```bash
node viewer/server.js --data-dir runs/Pixel
# open http://localhost:8787 — each horizon strip shows seen t -> predicted t+h
# -> actual t+h -> |error|, plus the EEG / attention / development panels.
```

## 13. Statistical referee

Route any before/after or candidate/baseline comparison through
`cognitive_runtime.training.statistical_evaluation` (CIs over N episodes) so a
regression is flagged with confidence intervals, not off a single noisy sample.